# Both-Success Analysis: Directed/Nudged Conditions

Find cases where **both TRUE and FALSE proofs compiled successfully** in directed/nudged conditions.

This indicates a formalization problem - logically, a statement and its negation cannot both be provable from consistent axioms.

In [1]:
import json
import re
import pandas as pd
from pathlib import Path
from collections import Counter

In [2]:
def load_results(dataset, model, condition, run):
    """Load results from a condition with iteration info."""
    path = Path(f'../results/{dataset}/{model}/{condition}_run{run}/results.jsonl')
    if not path.exists():
        return {}
    results = {}
    with open(path) as f:
        for line in f:
            entry = json.loads(line)
            case_idx = entry.get('case_idx')
            lv = entry.get('lean_verification', {})
            iterations = entry.get('iterations', [])
            
            # Get last iteration's lean verification
            last_iter_lv = None
            if iterations:
                last_iter = iterations[-1]
                last_iter_lv = last_iter.get('lean_verification', {})
            
            results[case_idx] = {
                'success': lv.get('success', False) if isinstance(lv, dict) else False,
                'prediction': entry.get('prediction'),
                'ground_truth': entry.get('ground_truth'),
                'correct': entry.get('correct'),
                'lean_code': entry.get('lean_code', ''),
                'errors': lv.get('errors', []) if isinstance(lv, dict) else [],
                'num_iterations': entry.get('num_iterations', 1),
                'iterations': iterations,
                'last_iter_lean_success': last_iter_lv.get('success', False) if isinstance(last_iter_lv, dict) else False
            }
    return results

def load_folio_data():
    """Load original FOLIO dataset."""
    with open('../data/folio/original/folio-validation.json') as f:
        return json.load(f)

folio_data = load_folio_data()
print(f"Loaded {len(folio_data)} FOLIO problems")

Loaded 203 FOLIO problems


---
## 1. Find Both-Success Cases

In [3]:
MODELS = ['gpt-5', 'deepseek']
DATASETS = ['folio', 'multilogieval']
CONDITION_PAIRS = [
    ('bidir_true', 'bidir_false'),
    ('spooky_true', 'spooky_false')
]

both_success_cases = []

for dataset in DATASETS:
    for model in MODELS:
        for true_cond, false_cond in CONDITION_PAIRS:
            for run in [1, 2, 3]:
                true_results = load_results(dataset, model, true_cond, run)
                false_results = load_results(dataset, model, false_cond, run)
                
                if not true_results or not false_results:
                    continue
                
                # Find cases present in both
                common_cases = set(true_results.keys()) & set(false_results.keys())
                
                for case_idx in common_cases:
                    true_success = True if true_results[case_idx]['prediction'].lower() in ['yes', 'true'] else False 
                    false_success = True if false_results[case_idx]['prediction'].lower() in ['no', 'false'] else False 
                    
                    if true_success and false_success:
                        both_success_cases.append({
                            'dataset': dataset,
                            'model': model,
                            'condition_pair': f'{true_cond}/{false_cond}',
                            'run': run,
                            'case_idx': case_idx,
                            'ground_truth': true_results[case_idx]['ground_truth'],
                            'true_pred': true_results[case_idx]['prediction'],
                            'false_pred': false_results[case_idx]['prediction'],
                            'true_correct': true_results[case_idx]['correct'],
                            'false_correct': false_results[case_idx]['correct'],
                            'true_code': true_results[case_idx]['lean_code'],
                            'false_code': false_results[case_idx]['lean_code'],
                            # Iteration info
                            'true_num_iterations': true_results[case_idx]['num_iterations'],
                            'false_num_iterations': false_results[case_idx]['num_iterations'],
                            'true_last_lean_success': true_results[case_idx]['last_iter_lean_success'],
                            'false_last_lean_success': false_results[case_idx]['last_iter_lean_success']
                        })

print(f"Total both-success cases: {len(both_success_cases)}")

Total both-success cases: 130


In [4]:
# Summary by dataset/model/condition
df = pd.DataFrame(both_success_cases)
df.head(3)

,dataset,model,condition_pair,run,case_idx,ground_truth,true_pred,false_pred,true_correct,false_correct,true_code,false_code,true_num_iterations,false_num_iterations,true_last_lean_success,false_last_lean_success
0,folio,gpt-5,bidir_true/bidir_false,1,75,Uncertain,True,False,False,False,axiom obj : Type\n\naxiom AtSchool : obj → Pro...,axiom obj : Type\n\naxiom Hannah : obj\n\n-- P...,1,1,True,True
1,folio,gpt-5,bidir_true/bidir_false,1,76,True,True,False,True,False,axiom obj : Type\naxiom Hannah : obj\n\naxiom ...,axiom obj : Type\naxiom Hannah : obj\n\naxiom ...,1,1,True,True
2,folio,gpt-5,bidir_true/bidir_false,1,77,False,True,False,False,True,axiom obj : Type\naxiom Hannah : obj\n\naxiom ...,axiom obj : Type\n\naxiom Hannah : obj\n\n-- P...,1,3,True,True


### Filtering Last Lean Success

In [5]:
final = df[df['true_last_lean_success'] & df['false_last_lean_success']]
final.shape[0]

118

## 2. Summary

In [6]:
final.groupby(['dataset', 'model', 'condition_pair']).agg({'case_idx' : 'nunique'})

case_idx
dataset       model    condition_pair                    
folio         deepseek bidir_true/bidir_false           7
                       spooky_true/spooky_false         7
              gpt-5    bidir_true/bidir_false           7
                       spooky_true/spooky_false        11
multilogieval deepseek bidir_true/bidir_false           8
                       spooky_true/spooky_false        11
              gpt-5    bidir_true/bidir_false           2
                       spooky_true/spooky_false         5

In [7]:
final.groupby(['dataset', 'model', 'condition_pair', 'ground_truth']).agg({'case_idx' : 'nunique'})

case_idx
dataset       model    condition_pair           ground_truth          
folio         deepseek bidir_true/bidir_false   False                2
                                                True                 3
                                                Uncertain            2
                       spooky_true/spooky_false False                2
                                                True                 3
                                                Uncertain            2
              gpt-5    bidir_true/bidir_false   False                2
                                                True                 3
                                                Uncertain            2
                       spooky_true/spooky_false False                3
                                                True                 5
                                                Uncertain            3
multilogieval deepseek bidir_true/bidir_false   no                   7
                                                yes                  1
                       spooky_true/spooky_false no                  10
                                                yes                  1
              gpt-5    bidir_true/bidir_false   no                   2
                       spooky_true/spooky_false no                   3
                                                yes                  2